# 01 — Tabular data preprocessing: multiclass sleep profiles

## Task 2 — From raw records to a validated multiclass table

This notebook is the preprocessing entry point for `notebooks/tabular/task2_sleep_profiles/`. It applies the workflow described in `docs/tabular_data_preprocessing.md` to the synthetic sleep-profile dataset.

The prediction target, `sleep_profile`, is **categorical and multiclass**. Preprocessing therefore has to preserve valid class names and verify them against the dataset metadata.

### Scope

This notebook performs structural preprocessing only:

- load metadata and raw data;
- validate the schema;
- assign analytical roles to columns;
- standardize column names and data types;
- inspect missing values and duplicates;
- validate binary categorical encodings;
- validate the multiclass target vocabulary;
- apply mechanical numerical checks;
- verify BMI consistency;
- export and reload the processed table.

It intentionally does **not** perform EDA, statistical comparison between classes, scaling for modelling, feature selection, train/test splitting, or machine-learning training.


## 1. Imports

The preprocessing stage needs only a small scientific Python stack. Keeping dependencies minimal makes the workflow easier to reproduce and helps students see which tools are actually required for each stage.


In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 150)


## 2. Locate the repository and define paths

Machine-specific absolute paths make notebooks difficult to share. We locate the repository root dynamically and derive every input and output path from it.

Raw data remain immutable. Processed data are written to a separate `processed_data/` directory.


In [2]:
def find_project_root(start: Path) -> Path:
    """Return the nearest parent directory containing the expected project folders."""
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate the repository root. Run the notebook from inside the project."
    )

PROJECT_ROOT = find_project_root(Path.cwd())

DATASET_NAME = "synthetic_sleep_profiles"
RAW_DIR = PROJECT_ROOT / "data" / DATASET_NAME / "raw_data"
PROCESSED_DIR = PROJECT_ROOT / "data" / DATASET_NAME / "processed_data"

RAW_DATA_PATH = RAW_DIR / "synthetic_sleep_profiles_1200.csv"
METADATA_PATH = RAW_DIR / "metadata.json"
PROCESSED_DATA_PATH = PROCESSED_DIR / "synthetic_sleep_profiles_1200_processed.csv"
REPORT_PATH = PROCESSED_DIR / "preprocessing_report.json"

print("Repository root:", PROJECT_ROOT)
print("Raw data:", RAW_DATA_PATH.relative_to(PROJECT_ROOT))
print("Metadata:", METADATA_PATH.relative_to(PROJECT_ROOT))
print("Processed output:", PROCESSED_DATA_PATH.relative_to(PROJECT_ROOT))


Repository root: /home/david/Desktop/depto_computacion/ml_2026/demos_machine_learning_2026
Raw data: data/synthetic_sleep_profiles/raw_data/synthetic_sleep_profiles_1200.csv
Metadata: data/synthetic_sleep_profiles/raw_data/metadata.json
Processed output: data/synthetic_sleep_profiles/processed_data/synthetic_sleep_profiles_1200_processed.csv


## 3. Read metadata before reading meaning into the columns

Column roles should come from the dataset contract, not from guesses based on names or pandas dtypes. For this task the metadata also declares the expected **multiclass target vocabulary**.


In [3]:
with METADATA_PATH.open("r", encoding="utf-8") as file:
    metadata = json.load(file)

required_metadata_keys = {
    "label",
    "label_type",
    "classes",
    "col_id",
    "categorical_column",
    "numerical_column",
}
missing_metadata_keys = required_metadata_keys - set(metadata)
if missing_metadata_keys:
    raise KeyError(f"Missing metadata keys: {sorted(missing_metadata_keys)}")

if metadata["label_type"] != "categorical_multiclass":
    raise ValueError("Task 2 expects label_type='categorical_multiclass'.")

TARGET_COLUMN = metadata["label"]
ID_COLUMN = metadata["col_id"]
TARGET_CLASSES = list(metadata["classes"])
CATEGORICAL_COLUMNS = list(metadata["categorical_column"])
NUMERICAL_COLUMNS = list(metadata["numerical_column"])

print("Identifier column:", ID_COLUMN)
print("Target column:", TARGET_COLUMN)
print("Label type:", metadata["label_type"])
print("Declared classes:", TARGET_CLASSES)
print("Categorical predictors:", len(CATEGORICAL_COLUMNS))
print("Numerical predictors:", len(NUMERICAL_COLUMNS))


Identifier column: participant_id
Target column: sleep_profile
Label type: categorical_multiclass
Declared classes: ['healthy_sleep', 'insomnia', 'sleep_apnea', 'circadian_disruption']
Categorical predictors: 6
Numerical predictors: 15


## 4. Read the raw table

Real files often encode missing information with several text tokens. Declaring common tokens at import time avoids treating strings such as `NA` or `unknown` as legitimate categories.


In [4]:
MISSING_TOKENS = [
    "NA",
    "N/A",
    "null",
    "None",
    "missing",
    "unknown",
]

raw_df = pd.read_csv(
    RAW_DATA_PATH,
    na_values=MISSING_TOKENS,
    keep_default_na=True,
)

print(f"Loaded {raw_df.shape[0]} rows and {raw_df.shape[1]} columns.")
display(raw_df.head())


Loaded 1200 rows and 23 columns.


,participant_id,age,sex_female,height_cm,weight_kg,bmi,sleep_duration_hours,sleep_efficiency_pct,sleep_latency_min,night_awakenings,resting_heart_rate_bpm,daily_steps,caffeine_mg_day,screen_time_evening_min,stress_score_0_10,social_jetlag_hours,bedtime_variability_min,shift_worker,regular_exercise,evening_alcohol,loud_snoring,anxiety_history,sleep_profile
0,SP0001,32,0,167.8,81.2,28.8,6.93,72.8,34.1,2,69,7285,320,78,5.0,1.59,98,1,0,0,0,0,circadian_disruption
1,SP0002,49,0,177.2,72.5,23.1,6.85,88.8,27.9,1,68,7383,62,158,6.1,2.44,120,1,1,0,0,0,circadian_disruption
2,SP0003,18,0,161.7,54.1,20.7,7.68,78.5,19.6,3,72,9446,322,113,0.5,1.41,42,0,1,0,0,0,healthy_sleep
3,SP0004,67,0,188.9,96.8,27.1,7.79,89.5,4.2,3,64,8444,103,16,3.1,0.54,13,0,1,1,0,0,healthy_sleep
4,SP0005,55,1,155.4,53.0,21.9,6.44,75.6,49.6,4,74,7097,134,71,6.9,1.12,48,0,1,0,0,0,insomnia


## 5. Validate the schema against metadata

Successful file loading does not guarantee that the correct table was loaded. We therefore compare observed and expected columns before making any transformation.


In [5]:
expected_columns_raw = {
    ID_COLUMN,
    TARGET_COLUMN,
    *CATEGORICAL_COLUMNS,
    *NUMERICAL_COLUMNS,
}
observed_columns = set(raw_df.columns)

missing_columns = sorted(expected_columns_raw - observed_columns)
unexpected_columns = sorted(observed_columns - expected_columns_raw)

display(pd.DataFrame({
    "check": ["Missing expected columns", "Unexpected columns"],
    "result": [missing_columns or "None", unexpected_columns or "None"],
}))

if missing_columns or unexpected_columns:
    raise ValueError(
        "The input schema does not match metadata. "
        f"Missing={missing_columns}, unexpected={unexpected_columns}"
    )


,check,result
0,Missing expected columns,None
1,Unexpected columns,None


## 6. Make column roles explicit

Identifiers, predictors, and targets have different meanings. Explicit roles reduce the chance of accidentally treating an identifier as a feature or modifying the target as though it were an ordinary predictor.


In [6]:
def column_role(column: str) -> str:
    if column == ID_COLUMN:
        return "identifier"
    if column == TARGET_COLUMN:
        return "target_categorical_multiclass"
    if column in CATEGORICAL_COLUMNS:
        return "categorical_predictor"
    if column in NUMERICAL_COLUMNS:
        return "numerical_predictor"
    return "unassigned"

role_table = pd.DataFrame({
    "column": raw_df.columns,
    "role": [column_role(column) for column in raw_df.columns],
    "raw_dtype": [str(raw_df[column].dtype) for column in raw_df.columns],
})
display(role_table)


,column,role,raw_dtype
0,participant_id,identifier,str
1,age,numerical_predictor,int64
2,sex_female,categorical_predictor,int64
3,height_cm,numerical_predictor,float64
4,weight_kg,numerical_predictor,float64
5,bmi,numerical_predictor,float64
6,sleep_duration_hours,numerical_predictor,float64
7,sleep_efficiency_pct,numerical_predictor,float64
8,sleep_latency_min,numerical_predictor,float64
9,night_awakenings,numerical_predictor,int64


## 7. Standardize column names

We use a conservative `snake_case` normalization. Any change is recorded and the metadata-derived role lists are normalized with exactly the same function.


In [7]:
def normalize_column_name(name: str) -> str:
    return (
        str(name)
        .strip()
        .lower()
        .replace(" ", "_")
        .replace("-", "_")
    )

normalized_columns = [normalize_column_name(column) for column in raw_df.columns]
if len(normalized_columns) != len(set(normalized_columns)):
    raise ValueError("Column-name normalization would create duplicated column names.")

rename_map = dict(zip(raw_df.columns, normalized_columns))
changed_names = {old: new for old, new in rename_map.items() if old != new}
print("Column names changed:", changed_names or "None")

df = raw_df.rename(columns=rename_map).copy()
ID_COLUMN = normalize_column_name(ID_COLUMN)
TARGET_COLUMN = normalize_column_name(TARGET_COLUMN)
CATEGORICAL_COLUMNS = [normalize_column_name(c) for c in CATEGORICAL_COLUMNS]
NUMERICAL_COLUMNS = [normalize_column_name(c) for c in NUMERICAL_COLUMNS]
FINAL_COLUMN_ORDER = normalized_columns
EXPECTED_COLUMNS = {ID_COLUMN, TARGET_COLUMN, *CATEGORICAL_COLUMNS, *NUMERICAL_COLUMNS}


Column names changed: None


## 8. Convert columns according to their declared roles

Type conversion is explicit rather than inferred implicitly:

- identifiers remain strings;
- numerical predictors become numeric;
- binary categorical predictors become nullable integers;
- the multiclass target remains a cleaned string label.

Any value that fails conversion is counted before the notebook continues.


In [8]:
conversion_failures = {}

df[ID_COLUMN] = df[ID_COLUMN].astype("string").str.strip()

for column in NUMERICAL_COLUMNS:
    original_non_missing = df[column].notna()
    converted = pd.to_numeric(df[column], errors="coerce")
    failed = original_non_missing & converted.isna()
    conversion_failures[column] = int(failed.sum())
    df[column] = converted

for column in CATEGORICAL_COLUMNS:
    original_non_missing = df[column].notna()
    converted = pd.to_numeric(df[column], errors="coerce")
    failed = original_non_missing & converted.isna()
    conversion_failures[column] = int(failed.sum())
    df[column] = converted.astype("Int64")

df[TARGET_COLUMN] = df[TARGET_COLUMN].astype("string").str.strip()

failure_table = pd.DataFrame({
    "column": list(conversion_failures),
    "conversion_failures": list(conversion_failures.values()),
})
failure_table = failure_table[failure_table["conversion_failures"] > 0]
display(failure_table if not failure_table.empty else pd.DataFrame({"conversion_failures": ["None"]}))

if any(conversion_failures.values()):
    raise ValueError("At least one value could not be converted to its expected data type.")


,conversion_failures
0,None


## 9. Check missing values

Missingness is identified here but not automatically imputed. Imputation can alter distributions and must later be designed with the train/test split in mind to avoid leakage.


In [9]:
missing_summary = df.isna().sum().rename("missing_values").to_frame()
missing_summary = missing_summary[missing_summary["missing_values"] > 0]
display(missing_summary if not missing_summary.empty else pd.DataFrame({"missing_values": ["None"]}))


,missing_values
0,None


## 10. Check exact duplicates and repeated identifiers

Exact row duplication and repeated identifiers are different problems. Exact copies may be removed mechanically; conflicting records sharing an identifier require explicit review and are not silently resolved.


In [10]:
exact_duplicate_count = int(df.duplicated().sum())
duplicated_id_count = int(df[ID_COLUMN].duplicated().sum())

print("Exact duplicated rows:", exact_duplicate_count)
print("Duplicated identifiers:", duplicated_id_count)

if exact_duplicate_count > 0:
    df = df.drop_duplicates().copy()
    print(f"Removed {exact_duplicate_count} exact duplicated rows.")

remaining_duplicated_ids = int(df[ID_COLUMN].duplicated().sum())
if remaining_duplicated_ids > 0:
    display(df.loc[df[ID_COLUMN].duplicated(keep=False), [ID_COLUMN]])
    raise ValueError("Repeated participant identifiers remain after exact deduplication.")


Exact duplicated rows: 0
Duplicated identifiers: 0


## 11. Validate binary categorical predictors

The categorical predictors in this teaching dataset are binary indicators. This is a **schema validation**, not a comparison between sleep-profile classes.


In [11]:
ALLOWED_BINARY_VALUES = {0, 1}
invalid_binary_values = {}

for column in CATEGORICAL_COLUMNS:
    observed = set(df[column].dropna().astype(int).unique().tolist())
    invalid = sorted(observed - ALLOWED_BINARY_VALUES)
    if invalid:
        invalid_binary_values[column] = invalid

print("Invalid binary encodings:", invalid_binary_values or "None")
if invalid_binary_values:
    raise ValueError(f"Invalid binary encodings found: {invalid_binary_values}")


Invalid binary encodings: None


## 12. Validate the multiclass target

A multiclass target should contain only labels declared by the dataset contract. We compare the observed vocabulary against `metadata.json` without yet analysing class frequencies or separability.


In [12]:
observed_classes = sorted(df[TARGET_COLUMN].dropna().unique().tolist())
declared_classes = sorted(TARGET_CLASSES)

missing_declared_classes = sorted(set(declared_classes) - set(observed_classes))
unexpected_classes = sorted(set(observed_classes) - set(declared_classes))

target_validation = pd.DataFrame({
    "property": ["Declared classes", "Observed classes", "Missing declared classes", "Unexpected classes"],
    "value": [declared_classes, observed_classes, missing_declared_classes or "None", unexpected_classes or "None"],
})
display(target_validation)

if missing_declared_classes or unexpected_classes:
    raise ValueError("Observed target classes do not match metadata.")


,property,value
0,Declared classes,"[circadian_disruption, healthy_sleep, insomnia..."
1,Observed classes,"[circadian_disruption, healthy_sleep, insomnia..."
2,Missing declared classes,None
3,Unexpected classes,None


## 13. Apply mechanical numerical sanity checks

Preprocessing should catch impossible encodings, not decide whether an observation is statistically unusual. Broad physical constraints are therefore used here; distributional outliers belong to EDA.


In [13]:
invalid_numeric_checks = {}

for column in NUMERICAL_COLUMNS:
    values = df[column].dropna()
    non_finite_count = int((~np.isfinite(values)).sum())
    if non_finite_count:
        invalid_numeric_checks[f"{column}: non-finite"] = non_finite_count

constraints = {
    "age": lambda s: (s < 0),
    "height_cm": lambda s: (s <= 0),
    "weight_kg": lambda s: (s <= 0),
    "bmi": lambda s: (s <= 0),
    "sleep_duration_hours": lambda s: (s < 0) | (s > 24),
    "sleep_efficiency_pct": lambda s: (s < 0) | (s > 100),
    "sleep_latency_min": lambda s: (s < 0),
    "night_awakenings": lambda s: (s < 0),
    "resting_heart_rate_bpm": lambda s: (s <= 0),
    "daily_steps": lambda s: (s < 0),
    "caffeine_mg_day": lambda s: (s < 0),
    "screen_time_evening_min": lambda s: (s < 0),
    "stress_score_0_10": lambda s: (s < 0) | (s > 10),
    "social_jetlag_hours": lambda s: (s < 0) | (s > 24),
    "bedtime_variability_min": lambda s: (s < 0),
}

for column, invalid_rule in constraints.items():
    invalid_count = int(invalid_rule(df[column].dropna()).sum())
    if invalid_count:
        invalid_numeric_checks[column] = invalid_count

print("Invalid numerical values:", invalid_numeric_checks or "None")
if invalid_numeric_checks:
    raise ValueError(f"Numerical sanity checks failed: {invalid_numeric_checks}")


Invalid numerical values: None


## 14. Check internal consistency of BMI

BMI is a derived variable, so it can be verified directly from height and weight. This is a deterministic consistency check, not a statistical association.


In [14]:
recomputed_bmi = df["weight_kg"] / (df["height_cm"] / 100.0) ** 2
bmi_consistent = np.allclose(
    df["bmi"].to_numpy(dtype=float),
    recomputed_bmi.to_numpy(dtype=float),
    atol=0.1,
    rtol=0,
    equal_nan=True,
)

display(pd.DataFrame({
    "derived_variable": ["bmi"],
    "consistent_with_source_columns": [bmi_consistent],
}))

if not bmi_consistent:
    raise ValueError("BMI is inconsistent with height and weight.")


,derived_variable,consistent_with_source_columns
0,bmi,True


## 15. Final structural validation

Before export, the processed table must still satisfy the intended contract. Assertions make these expectations executable and visible.


In [15]:
df = df.loc[:, FINAL_COLUMN_ORDER].reset_index(drop=True)

assert list(df.columns) == FINAL_COLUMN_ORDER, "Column order changed unexpectedly."
assert set(df.columns) == EXPECTED_COLUMNS, "Final schema differs from metadata."
assert df[ID_COLUMN].notna().all(), "Missing identifiers remain."
assert df[ID_COLUMN].is_unique, "Identifiers are not unique."
assert df[TARGET_COLUMN].notna().all(), "Missing target labels remain."
assert df.duplicated().sum() == 0, "Exact duplicate rows remain."
assert not any(conversion_failures.values()), "Data-type conversion failures remain."
assert not invalid_binary_values, "Invalid categorical encodings remain."
assert not invalid_numeric_checks, "Invalid numerical values remain."
assert set(df[TARGET_COLUMN].unique()) == set(TARGET_CLASSES), "Target vocabulary changed."
assert bmi_consistent, "BMI consistency check failed."

final_validation = pd.DataFrame({
    "property": ["Rows", "Columns", "Exact duplicate rows", "Duplicated identifiers", "Missing values", "Target classes"],
    "value": [
        df.shape[0], df.shape[1], int(df.duplicated().sum()), int(df[ID_COLUMN].duplicated().sum()),
        int(df.isna().sum().sum()), len(df[TARGET_COLUMN].unique()),
    ],
})
display(final_validation)


,property,value
0,Rows,1200
1,Columns,23
2,Exact duplicate rows,0
3,Duplicated identifiers,0
4,Missing values,0
5,Target classes,4


## 16. Export the processed dataset and a machine-readable report

The raw table is never overwritten. The report records what entered and left preprocessing so downstream notebooks can reuse the same column roles instead of redefining them manually.


In [16]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
df.to_csv(PROCESSED_DATA_PATH, index=False)

preprocessing_report = {
    "dataset": DATASET_NAME,
    "task_type": "multiclass_classification",
    "label_type": metadata["label_type"],
    "input_file": str(RAW_DATA_PATH.relative_to(PROJECT_ROOT)),
    "metadata_file": str(METADATA_PATH.relative_to(PROJECT_ROOT)),
    "output_file": str(PROCESSED_DATA_PATH.relative_to(PROJECT_ROOT)),
    "input_shape": [int(raw_df.shape[0]), int(raw_df.shape[1])],
    "output_shape": [int(df.shape[0]), int(df.shape[1])],
    "identifier_column": ID_COLUMN,
    "target_column": TARGET_COLUMN,
    "target_classes": TARGET_CLASSES,
    "numerical_columns": NUMERICAL_COLUMNS,
    "categorical_columns": CATEGORICAL_COLUMNS,
    "exact_duplicate_rows_found": exact_duplicate_count,
    "duplicated_identifiers_found": duplicated_id_count,
    "missing_values_after_preprocessing": int(df.isna().sum().sum()),
    "bmi_consistency_check": bool(bmi_consistent),
}

with REPORT_PATH.open("w", encoding="utf-8") as file:
    json.dump(preprocessing_report, file, indent=2)

print("Processed dataset saved to:", PROCESSED_DATA_PATH.relative_to(PROJECT_ROOT))
print("Preprocessing report saved to:", REPORT_PATH.relative_to(PROJECT_ROOT))


Processed dataset saved to: data/synthetic_sleep_profiles/processed_data/synthetic_sleep_profiles_1200_processed.csv
Preprocessing report saved to: data/synthetic_sleep_profiles/processed_data/preprocessing_report.json


## 17. Reload the exported table

A successful write call does not prove that the serialized file can be reconstructed correctly. We therefore reload the CSV and compare it against the in-memory processed DataFrame.


In [17]:
reloaded_df = pd.read_csv(PROCESSED_DATA_PATH)
pd.testing.assert_frame_equal(
    df.reset_index(drop=True),
    reloaded_df.reset_index(drop=True),
    check_dtype=False,
)
print("Round-trip validation: PASS")
print("Reloaded shape:", reloaded_df.shape)


Round-trip validation: PASS
Reloaded shape: (1200, 23)


## 18. What was intentionally not done here?

The processed table is now structurally ready for exploratory analysis, but preprocessing is not EDA.

We have **not** examined class-specific distributions, class balance, group differences, correlations, PCA, nonlinear projections, statistical tests, effect sizes, predictive performance, feature importance, or train/test splits.

Those questions belong to the next stage. Keeping the boundary explicit makes the workflow easier to reason about and reduces the risk of information leakage later.
